[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch05_patterns.ipynb)

# Chapter 5 — Patterns and Anti-Patterns

The anti-pattern measured here is the one everyone has tried: making the
instruction louder.

Note that each condition changes exactly **one** thing. An earlier draft
changed two — capitals *and* an added "important" — which measures nothing,
and is the same confound §14.6 warns about.

In [1]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [2]:
from peb import ask
from peb.client import Blocked

RUNS = 10

BASE = ("Summarise the paragraph below in one sentence of at most 15 words.\n\n"
        "The depot moved 412 pallets in March, up from 388 in February, "
        "though two of the four loading bays were closed for resurfacing.")

CONDITIONS = {
    "plain":    BASE,
    "shouting": BASE.replace("at most 15 words", "AT MOST 15 WORDS"),
    "urgent":   BASE.replace("Summarise", "IMPORTANT! Summarise"),
    # This condition is currently refused by a platform filter citing "reverse
    # engineering or duplicating model outputs" -- the WORKING:/ANSWER: framing
    # reads as an attempt to extract raw reasoning. It is kept, and the block is
    # reported as its own outcome, because a reader who writes this prompt will
    # hit the same wall. Note that §4.4's "Show your working, then give the
    # answer" is NOT affected; it runs normally.
    "format":   BASE + "\n\nBegin your reply with the line 'WORKING:' and show "
                       "your steps, then a line 'ANSWER:' with just the number.",
}

In [3]:
def within_limit(text):
    body = text.split("ANSWER:")[-1] if "ANSWER:" in text else text
    return len(body.split()) <= 15


for label, prompt in CONDITIONS.items():
    ok = blocked = 0
    for _ in range(RUNS):
        try:
            ok += within_limit(ask(prompt, max_tokens=4000)[0])
        except Blocked:
            blocked += 1
    answered = RUNS - blocked
    note = f"   ({blocked}/{RUNS} blocked upstream)" if blocked else ""
    print(f"{label:<10} {ok}/{answered or '-'} inside the limit{note}")

print("\nEmphasis is not an instruction. Structure is.")

plain      5/10 inside the limit


shouting   9/10 inside the limit


urgent     9/10 inside the limit


format     0/- inside the limit   (10/10 blocked upstream)

Emphasis is not an instruction. Structure is.
